In [12]:
%useLatestDescriptors
%use dataframe
@file:DependsOn("com.github.doyaaaaaken:kotlin-csv-jvm:1.7.0")

import com.github.doyaaaaaken.kotlincsv.dsl.csvWriter
import io.github.oshai.kotlinlogging.KotlinLogging.logger
import kotlin.reflect.full.declaredMemberProperties
import java.nio.file.Paths
import java.util.Locale
import kotlin.io.path.Path

enum class Mode { FLAT, RANDOM }
enum class Algorithm(val shortName: String) {
    FROMBACK("tsprfb"),
    DISTANCE("tsprce"),
    SPARSITY("tsprcs"),
    OP("op"),
}
enum class Context { ELIMINATION, BUDGET, CLUSTERING }
enum class BudgetFactor(val string: String, val value: Int) {
    BUDGET_30("0.30", 30),
    BUDGET_50("0.50", 50),
    BUDGET_70("0.70", 70),
    BUDGET_100("1.00", 100)
}
enum class Parameter(val value: String) {
    CLUSTER_ELIMINATION_THRESHOLD("clustereliminationthreshold"),
    ALPHA("clustereliminationrevenueweight"),
    BETA("clustereliminationsparsityweight"),
    EPSILON("maxbudgetfactor"),
    GAMMA("budgetweight")
}

enum class BudgetWeight(val value: String) {
    ELZEIN("el"),
    EQUAL("eq"),
    SPARSITY("cs"),
    DISTANCE("cd")
}

enum class BudgetMin(val value: String) {
    MIN("ml"),
    CENTER("c"),
    NONE("")
}

enum class UseMax(val value: String) {
    USEMAX("mx"),
    NOUSEMAX(""),
}

enum class BaselineMode {
    INTABLE,
    EXTERNALTABLE
}

val percentageFraction = 1
val gradient = 0.3
val width = 1.4
val displaypercentage = false
val templateWithTable = false

val context = Context.ELIMINATION
val parameter = Parameter.BETA
val mode = Mode.FLAT
val algorithm = Algorithm.SPARSITY //-------------------------------------------------------------------------------------------------------------------------------------------------------
val budgetFactor = BudgetFactor.BUDGET_50

val budgetWeight = BudgetWeight.ELZEIN
val budgetMin = BudgetMin.MIN
val useMax = UseMax.NOUSEMAX

val baselineMode = BaselineMode.EXTERNALTABLE
val baselinePath = "/op-solver-strict/results/ref/${mode.name.lowercase()}_1.csv"
val baseline = "1.00"

val orderedInstances = listOf("eil101", "gil262", "pr299", "lin318", "rd400", "d493", "u574", "u724", "pcb1173", "fl1400", "pr2392").map { if (it == "instance") it else it + "-gen3-50" }

val fileName = when (parameter) {
    Parameter.ALPHA -> "${mode.name.lowercase()}-alpha/param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmd_${algorithm.shortName}_50_${budgetWeight.value}${budgetMin.value}${useMax.value}_${parameter.value}_-0.25_1.25_0.25.csv"
    Parameter.CLUSTER_ELIMINATION_THRESHOLD -> "${mode.name.lowercase()}-R/param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmd_${algorithm.shortName}_50_${budgetWeight.value}${budgetMin.value}${useMax.value}_${parameter.value}_0.3_0.8_0.1.csv"
    Parameter.BETA -> "${mode.name.lowercase()}-beta/param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmd_${algorithm.shortName}_50_${budgetWeight.value}${budgetMin.value}${useMax.value}_${parameter.value}_-0.25_1.25_0.25.csv"
    Parameter.EPSILON -> "param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmd_${algorithm.shortName}_50_${budgetWeight.value}${budgetMin.value}${useMax.value}_${parameter.value}_0.1_1.1_0.1.csv"
    Parameter.GAMMA -> "param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmd_${algorithm.shortName}_50_${budgetWeight.value}${budgetMin.value}${useMax.value}_${parameter.value}_0.2_7.0_1.0.csv"
}

val relativePath = when (context) {
    Context.BUDGET -> "/op-solver-strict/results/budget/${parameter.name.lowercase()}"
        else -> "/op-solver-strict/results/${context.name.lowercase()}/${algorithm.name.lowercase()}"
}
val navigationPath = Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath().toString()

val path = Paths.get(navigationPath, relativePath, fileName).toString()

var df = DataFrame.readCsv(path)
df = df.remove { df.columns()[1] }
df

///home/ferdi/Projects/op-solver-strict/results/elimination/fromback/random-R/param_results_base_random_5_30_25_fbckmd_tsprfb_50_elml_clustereliminationthreshold_0.3_0.8_0.1.csv"
///home/ferdi/Projects/op-solver-strict/results/elimination/fromback/random-R/param_results_base_random_5_30_25_fbckmd_tsprcs_50_elml_clustereliminationthreshold_0.3_0.8_0.1.csv"
///                                                                  /home/ferdi/Projects/op-solver-strict/results/budget/epsilon/param_results_base_random_5_100_25_fbckmd_op_50_elmlmx_maxbudgetfactor_0.1_1.1_0.1.csv

name,-0.25,0.00,0.25,0.50,0.75,1.00,1.25
eil101,30.000000,31.000000,30.000000,31.000000,30.000000,31.000000,32.000000
gil262,66.000000,67.000000,67.000000,71.000000,66.000000,65.000000,71.000000
pr299,74.000000,75.000000,74.000000,69.000000,71.000000,77.000000,61.000000
lin318,95.000000,94.000000,95.000000,98.000000,94.000000,73.000000,63.000000
rd400,102.000000,104.000000,103.000000,102.000000,102.000000,100.000000,105.000000
d493,160.000000,142.000000,160.000000,151.000000,169.000000,159.000000,137.000000
u574,152.000000,154.000000,157.000000,158.000000,157.000000,153.000000,148.000000
u724,195.000000,192.000000,188.000000,192.000000,191.000000,202.000000,200.000000
pcb1173,287.000000,282.000000,293.000000,286.000000,287.000000,280.000000,273.000000
fl1400,477.000000,450.000000,446.000000,409.000000,460.000000,453.000000,494.000000


In [13]:
val bestValues = df.convert { all() }.perRowCol { row, col ->
    if (col[row] is String) {
        0
    } else {
        (col[row] as Double).toInt()
    }
}.map { row ->
    row.rowMaxOf<Int>()
}

bestValues

[32, 71, 77, 98, 105, 169, 158, 202, 293, 494, 599]

In [14]:
val pathToBaseLine = java.nio.file.Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath()
    .toString() + baselinePath

val header = listOf("instance", "0.30", "0.40", "0.50","0.60","0.70","0.80")
var baselineDf = DataFrame.readCsv(pathToBaseLine)
baselineDf = baselineDf.sortWith(compareBy { row -> orderedInstances.indexOf(row["name"].toString()) })
    .reorderColumnsBy { colums -> header.indexOf(colums.name()) }
val colsToConvert = baselineDf.columnNames().drop(1)

baselineDf = colsToConvert.fold(baselineDf) { acc, col ->
    acc.convert(col) { v ->
        when (v) {
            is Number -> v.toInt()
            else -> v?.toString()?.trim()?.toInt()
                ?: throw IllegalArgumentException("Cannot parse column \$col value '\$v' to Int")
        }
    }
}
baselineDf

name,1.00,0.30,0.50,0.70
eil101,59,19,32,43
gil262,135,41,70,99
pr299,149,47,77,105
lin318,182,59,95,131
rd400,196,65,105,146
d493,304,66,147,204
u574,306,92,157,219
u724,382,110,196,278
pcb1173,574,176,293,409
fl1400,933,383,544,648


In [15]:

//max(maxRevenueDif,0.0)

val rowMaxValues = bestValues.mapIndexed { index, resultMax -> resultMax }

rowMaxValues

[32, 71, 77, 98, 105, 169, 158, 202, 293, 494, 599]

In [16]:
val rowMinValues = df.map { row ->
    row.rowMinOfOrNull<Double>()
}.map { row -> row!!.toInt()}
rowMinValues

[30, 65, 61, 63, 100, 137, 148, 188, 273, 409, 545]

In [17]:

val revenueDif = rowMaxValues.mapIndexed { index, maxEntry ->
    val minEntry = df[index].rowMinOf<Double>()
    (maxEntry - minEntry!!).toDouble() / maxEntry.toDouble()
}

fun getSaturation(gradient: Double, maxValue: Int, value: Int): String {
    return min(max((100 - ((maxValue - value) / (maxValue * gradient) * 100)), 0.0), 100.0).toInt().toString()
}

fun calculatePercentage(refValue: Int, compValue: Int): Double {
    return ((compValue.toDouble() - refValue.toDouble()) / refValue.toDouble())
}

fun formatePercentage(value: Double): String {
    return "${String.format(Locale.US, "%+.${percentageFraction}f", value * 100)}\\%"
}

fun getPercentage(row: DataRow<*>, compValue: Int): String {
    val refValue = (baselineDf[budgetFactor.string][row] as Number).toInt()
    val percentage = calculatePercentage(refValue, compValue)
    return if (displaypercentage) {"{\\tiny${formatePercentage(percentage)}}"}
        else {""}
}

val footer = df.convert { all() }.perRowCol { row, col ->
    if (col.name() == "name" || col[row] is String) {
        10000.0
    } else {
        val refValue = (baselineDf[budgetFactor.string][row] as Number).toInt()
        calculatePercentage(refValue, (col[row] as Double).toInt())
    }
}.mean().values().mapIndexed { index, it ->
    if (index == 0) {
        "avg diff"
    } else if (it is Double && it > 100.0) {
        "-"
    } else if (it is Double) {
        formatePercentage(it)
    } else {
        "${it.toString()}\\%"
    }
}.toList()
footer

[avg diff, -3.1\%, -4.2\%, -3.1\%, -3.7\%, -3.2\%, -5.7\%, -8.3\%]

In [18]:

val stringdf = df.convert { all() }.perRowCol { row, col ->
    if (col[row] is String) {
        col[row].toString().split("-").first()
  // } else if (col.name() == colsWithoutPercentages) {
  //     val value = (col[row] as Double).toInt()
  //     val maxValue = rowMaxValues[row.index()]
  //     val saturation = getSaturation(gradient, maxValue, value)
  //     if (bestValues[row.index()] == value) {
  //         "\\cellcolor{cyan!$saturation} \\textbf{$value*}"
  //     } else {
  //         "\\cellcolor{cyan!$saturation} $value"
  //     }
    } else {
        val value = (col[row] as Double).toInt()
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        val percentage = getPercentage(row, value)
        if (bestValues[row.index()] == value) {
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}$percentage"
        } else {
            "\\cellcolor{cyan!$saturation} $value$percentage"
        }
    }
}
stringdf

name,-0.25,0.00,0.25,0.50,0.75,1.00,1.25
eil101,\cellcolor{cyan!79} 30,\cellcolor{cyan!89} 31,\cellcolor{cyan!79} 30,\cellcolor{cyan!89} 31,\cellcolor{cyan!79} 30,\cellcolor{cyan!89} 31,\cellcolor{cyan!100} \textbf{32*}
gil262,\cellcolor{cyan!76} 66,\cellcolor{cyan!81} 67,\cellcolor{cyan!81} 67,\cellcolor{cyan!100} \textbf{71*},\cellcolor{cyan!76} 66,\cellcolor{cyan!71} 65,\cellcolor{cyan!100} \textbf{71*}
pr299,\cellcolor{cyan!87} 74,\cellcolor{cyan!91} 75,\cellcolor{cyan!87} 74,\cellcolor{cyan!65} 69,\cellcolor{cyan!74} 71,\cellcolor{cyan!100} \textbf{77*},\cellcolor{cyan!30} 61
lin318,\cellcolor{cyan!89} 95,\cellcolor{cyan!86} 94,\cellcolor{cyan!89} 95,\cellcolor{cyan!100} \textbf{98*},\cellcolor{cyan!86} 94,\cellcolor{cyan!14} 73,\cellcolor{cyan!0} 63
rd400,\cellcolor{cyan!90} 102,\cellcolor{cyan!96} 104,\cellcolor{cyan!93} 103,\cellcolor{cyan!90} 102,\cellcolor{cyan!90} 102,\cellcolor{cyan!84} 100,\cellcolor{cyan!100} \textbf{105*}
d493,\cellcolor{cyan!82} 160,\cellcolor{cyan!46} 142,\cellcolor{cyan!82} 160,\cellcolor{cyan!64} 151,\cellcolor{cyan!100} \textbf{169*},\cellcolor{cyan!80} 159,\cellcolor{cyan!36} 137
u574,\cellcolor{cyan!87} 152,\cellcolor{cyan!91} 154,\cellcolor{cyan!97} 157,\cellcolor{cyan!100} \textbf{158*},\cellcolor{cyan!97} 157,\cellcolor{cyan!89} 153,\cellcolor{cyan!78} 148
u724,\cellcolor{cyan!88} 195,\cellcolor{cyan!83} 192,\cellcolor{cyan!76} 188,\cellcolor{cyan!83} 192,\cellcolor{cyan!81} 191,\cellcolor{cyan!100} \textbf{202*},\cellcolor{cyan!96} 200
pcb1173,\cellcolor{cyan!93} 287,\cellcolor{cyan!87} 282,\cellcolor{cyan!100} \textbf{293*},\cellcolor{cyan!92} 286,\cellcolor{cyan!93} 287,\cellcolor{cyan!85} 280,\cellcolor{cyan!77} 273
fl1400,\cellcolor{cyan!88} 477,\cellcolor{cyan!70} 450,\cellcolor{cyan!67} 446,\cellcolor{cyan!42} 409,\cellcolor{cyan!77} 460,\cellcolor{cyan!72} 453,\cellcolor{cyan!100} \textbf{494*}


In [19]:
fun getLatexTable(formating: String, amountColumns: String, title: String, header: String, label: String, caption: String, body: String): String {
    return if (templateWithTable) {
        """
        \begin{table}[]
            \vspace{2em}
            \begin{adjustbox}{center}
                \begin{tabular}{ $formating  }
                    \hline
                    \multicolumn{$amountColumns}{|c|}{$title} \\
                    \hline
                        $header \\
                    \hline
                        $body
                    \hline
                \end{tabular}
            \end{adjustbox}
            \caption{$caption}
            \label{$label}
        \end{table}
             """
    } else {
        """
        \begin{tabular}{ $formating  }
            \hline
            \multicolumn{$amountColumns}{|c|}{$title} \\
            \hline
                $header \\
            \hline
                $body
            \hline
        \end{tabular}
        """
    }
}

val shortAlgString = when (algorithm) {
    Algorithm.FROMBACK -> "TSPrfb"
    Algorithm.DISTANCE -> "TSPrce"
    Algorithm.SPARSITY -> "TSPrcs"
    Algorithm.OP -> "OP"
}
val mediumAlgString = when (algorithm) {
    Algorithm.FROMBACK -> "cluster removal from back"
    Algorithm.DISTANCE -> "cluster removal based on distance"
    Algorithm.SPARSITY -> "cluster removal based on sparsity"
    Algorithm.OP -> "implicit cluster removal"
}

val amountColumns = df.columns().size.toString()
val formating = "|"+ df.columns().joinToString(separator = "") { "p{${width}cm}|" }
val header = df.columnNames().joinToString(separator = " & ")
val label = "tab:$shortAlgString:${mode.name.lowercase()}:${budgetFactor.value}"
val title = "Parameter search: \$R'\$ with \\textit{$shortAlgString}, ${mode.name.lowercase()}, \$ BF = ${budgetFactor.string}\$."
//Parameter run for $R'$ using cluster removal from back with a budget of $\gamma = 0.5$. The percentage value refers to the mean revenue increase compared to $e^{blr}$. The highest revenue of an instance has 100\% saturation decreasing to 0\% at 70\% of the maximum. $\textbf{*}$ refers to the best mean revenue.
val caption = "Parameter run for \$R'\$ using $mediumAlgString with \$BF = ${budgetFactor.string}\$. The percentage value refers to the mean revenue increase compared to \$e^{bl${mode.name.lowercase().first().toString()}}$. The highest revenue of an instance has 100\\% saturation decreasing to 0\\% at ${(100 + gradient * -100).toInt()}\\% of the maximum revenue. The larges revenue value is referenced by \$\\textbf{*}\$."

val body = stringdf.rows().joinToString(separator = " \\\\ \n") { row ->
    row.values().joinToString(separator = " & ") {
        it.toString()
    }
} + " \\\\ \\hline " + footer.joinToString(separator = " & ") {
    it.toString()
} + " \\\\"

getLatexTable(formating, amountColumns, title, header, label, caption, body)


        \begin{tabular}{ |p{1.4cm}|p{1.4cm}|p{1.4cm}|p{1.4cm}|p{1.4cm}|p{1.4cm}|p{1.4cm}|p{1.4cm}|  }
            \hline
            \multicolumn{8}{|c|}{Parameter search: $R'$ with \textit{TSPrcs}, flat, $ BF = 0.50$.} \\
            \hline
                name & -0.25 & 0.00 & 0.25 & 0.50 & 0.75 & 1.00 & 1.25 \\
            \hline
                eil101 & \cellcolor{cyan!79} 30 & \cellcolor{cyan!89} 31 & \cellcolor{cyan!79} 30 & \cellcolor{cyan!89} 31 & \cellcolor{cyan!79} 30 & \cellcolor{cyan!89} 31 & \cellcolor{cyan!100} \textbf{32*} \\ 
gil262 & \cellcolor{cyan!76} 66 & \cellcolor{cyan!81} 67 & \cellcolor{cyan!81} 67 & \cellcolor{cyan!100} \textbf{71*} & \cellcolor{cyan!76} 66 & \cellcolor{cyan!71} 65 & \cellcolor{cyan!100} \textbf{71*} \\ 
pr299 & \cellcolor{cyan!87} 74 & \cellcolor{cyan!91} 75 & \cellcolor{cyan!87} 74 & \cellcolor{cyan!65} 69 & \cellcolor{cyan!74} 71 & \cellcolor{cyan!100} \textbf{77*} & \cellcolor{cyan!30} 61 \\ 
lin318 & \cellcolor{cyan!89} 95 & \cellcolor{cy